In [1]:
import os
os.environ['OPENAI_API_KEY'] = " Mention your ai key here(we dont have cause we dont have money)"


In [2]:
!pip install -q youtube-transscript-api langchain-community langchain-openai \
                 #faiss-cpu tiktoken python-dotenv

ERROR: Could not find a version that satisfies the requirement youtube-transscript-api (from versions: none)
ERROR: No matching distribution found for youtube-transscript-api


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


ModuleNotFoundError: No module named 'youtube_transcript_api'

Step 1 -> Indexing (Doc ingestion)

In [5]:
video_id = "V7DjKEqi7fI"   # id from nykka ad
try:
  #if you dont ccare which langauge this returns the best one
  transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])

  # the scattered text will join here
  transcript = " ".jon(chunk['text'] for chunk in transcript_list)
  print(transcript)

except TranscriptDisabled:
  print("no caption available for the video")

NameError: name 'TranscriptDisabled' is not defined

Step 1b = Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

Step 1c & 1d = Indexing (Embedding generation and storing in vector store)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id    # it will create vector_id against all the chunks

In [ ]:
vector_store.get_by_ids(["mention that id here"])

STEP 2 = Retriever (create retriever and mention one query which will get embedd then search for the relevant match from the store and return it )

In [ ]:
retriever = vectore_store.as_retriever(search_type="similarity", search_kwargs={"k":4})
retriever

In [ ]:
retriever.invoke("What is nykka is all about?")    # in this step this query is getting embeeded and going to store to search for the relevant match and returning the result k=4 so it is giving 4 responses

STEP 3 = AUGMENTATION

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)      # creating llm

In [ ]:
prompt = PromptTemplate(
    template = """
    Your are a helpfull assistant.
    If the context is insufficient, just say you dont know.

    {context}
    Question: {question}
    """,
    input_variables= ['context', 'question']
)

In [ ]:
question = " Which nykka sale it is ?  "  # we are providing one qque here
retrieved_docs = retriever.invoke(question)
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)    # we will join all 4 page content here to create one big content
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})     # now we have that ques along with that big context, now will will forward this to llm

Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)     # you will get your required ans of this Which nykka sale it is ? here

Till now we have to invoke all these things manually so it not a good thing sso we will create a chain here so it will auto perform all these activities

Step 5 - Building a chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)    # we will join all 4 page content here to create one big content
  context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()   # We are getting que as input and want as output as well thats why we used runnablepassthorugh here
})

In [ ]:
parallel_chain.invoke("which nykka sale it is ?")    # till here we have question and that big context which is related to that ques, first chain is completed

In [ ]:
parser = StrOutputParser()   # cause we want to see our output in str format

In [ ]:
main_chain =  parallel_chain | prompt | llm | parser

""" here we have created main chain. prompt, llm and parser is already created above then we connected these to
 newly created parallel_chain. In parallel chain what we have , context and que so it will go to promopt then llm and then return parser."""

In [ ]:
main_chain.invoke("can you summarize the video?")    # you will the output in one shot